# Khảo sát dữ liệu M5 — H01 (Hùng)

Chạy notebook từ thư mục gốc của repo. Đặt 5 file CSV gốc trong `data/raw/`. Xem kết quả đầy đủ trong `reports/Hung/data_profile.md` và danh sách sản phẩm MVP đề xuất trong `reports/Hung/m5_mvp_items.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

raw = Path('data/raw')
files = ['sales_train_validation.csv', 'sales_train_evaluation.csv', 'calendar.csv', 'sell_prices.csv', 'sample_submission.csv']
pd.DataFrame([{'Tên file': name, 'Dung lượng (MiB)': round((raw / name).stat().st_size / 1024**2, 2), 'Số cột': len(pd.read_csv(raw / name, nrows=0).columns)} for name in files])

In [ ]:
ids = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
metadata = pd.read_csv(raw / 'sales_train_evaluation.csv', usecols=ids)
print('Số chuỗi:', len(metadata), 'Số sản phẩm:', metadata.item_id.nunique(), 'Số cửa hàng:', metadata.store_id.nunique())
display(metadata.store_id.value_counts().sort_index().rename('Số chuỗi').to_frame())
display(metadata.isna().sum().rename('Số giá trị thiếu').to_frame())
calendar = pd.read_csv(raw / 'calendar.csv')
display(calendar[['d', 'date', 'wm_yr_wk', 'event_name_1', 'snap_CA']].head())

In [ ]:
selected = pd.read_csv('reports/Hung/m5_mvp_items.csv')
item_ids = set(selected.item_id)
parts = []
for chunk in pd.read_csv(raw / 'sales_train_evaluation.csv', chunksize=1000):
    rows = chunk.loc[chunk.store_id.eq('CA_1') & chunk.item_id.isin(item_ids)]
    if not rows.empty:
        parts.append(rows)
mvp = pd.concat(parts, ignore_index=True)
days = [column for column in mvp if column.startswith('d_')]
sales = mvp[days]
print('Số chuỗi MVP:', len(mvp), 'Giá trị thiếu:', int(sales.isna().sum().sum()), 'Tỷ lệ doanh số bằng 0:', f'{(sales == 0).sum().sum() / sales.size:.2%}')
daily = sales.sum(axis=0).rename_axis('d').reset_index(name='so_luong_ban')
daily = daily.merge(calendar[['d', 'date']], on='d', validate='one_to_one')
daily['date'] = pd.to_datetime(daily['date'])
daily.plot(x='date', y='so_luong_ban', figsize=(12, 4), title='CA_1 — Doanh số mỗi ngày của 300 sản phẩm đã chọn', legend=False)
plt.xlabel('Ngày')
plt.ylabel('Số lượng bán')
plt.show()

Biểu đồ và tỷ lệ doanh số bằng 0 chỉ phản ánh 300 chuỗi sản phẩm–cửa hàng đã chọn. Cần thống nhất phạm vi này với nhóm trước H02. Giữ các ngày có doanh số bằng 0 vì đó là quan sát hợp lệ.